In [ ]:
# ============================================
# MiniLLM - Qwen3-0.6B
# Python + PyTorch + Transformers + Gradio
# ============================================

import time
start_time = time.time()

# 1. Install dependencies
!pip -q install -U transformers accelerate gradio

# 2. Import libraries
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

# 3. Check GPU
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

# 4. Load Qwen model
MODEL_ID = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto"
)

model.eval()

print("Loaded:", MODEL_ID)
print("Model loading time:",
      round(time.time() - start_time, 2), "seconds")

# 5. System prompt
SYSTEM_PROMPT = (
    "You are MiniLLM, a helpful and concise AI assistant. "
    "Explain technical topics clearly and use examples when useful."
)

# 6. LLM response function
first_response = True

def respond(message, history):

    global first_response
    response_start = time.time()

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ]

    # Support both old and new Gradio history formats
    for item in (history or [])[-6:]:

        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content", "")

            if role in ("user", "assistant") and isinstance(content, str):
                messages.append({
                    "role": role,
                    "content": content
                })

        elif isinstance(item, (list, tuple)) and len(item) == 2:
            user_message, assistant_message = item

            if user_message:
                messages.append({
                    "role": "user",
                    "content": str(user_message)
                })

            if assistant_message:
                messages.append({
                    "role": "assistant",
                    "content": str(assistant_message)
                })

    # Add current question
    messages.append({
        "role": "user",
        "content": message
    })

    # Format prompt
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    # Tokenize input
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    # Generate answer
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=True,
            temperature=0.7,
            top_p=0.8,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode answer
    new_tokens = output[0][inputs["input_ids"].shape[-1]:]

    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    # Measure response time
    elapsed = time.time() - response_start

    print(f"Response time: {elapsed:.2f} seconds")

    if first_response:
        print(f"First response: {elapsed:.2f} seconds")
        first_response = False

    return answer

# 7. Launch Gradio chatbot
demo = gr.ChatInterface(
    fn=respond,
    title="MiniLLM - Qwen3-0.6B",
    description="A lightweight LLM chatbot powered by Qwen.",
    examples=[
        "What is an LLM?",
        "Explain artificial intelligence.",
        "Write a Python Hello World program."
    ]
)

print("Application initialization time:",
      round(time.time() - start_time, 2), "seconds")

demo.launch(share=True, debug=True)
